# 02 — Estrazione strutturata con LMM (Blocco A)

Per ogni coppia (paese, trimestre) legge i documenti grezzi raccolti nel notebook 01 (PDF renderizzati in immagini + testo Wikipedia/CFR/CISA) e produce un JSON conforme a `config/extraction_schema.json`, salvato in `data/processed/extracted_json/<ISO3>/`.

**Modello**: Qwen2.5-VL:7b via Ollama, servito in locale (Mac per sviluppo/test, job PBS su Leonardo per l'esecuzione completa). La generazione e' vincolata allo JSON Schema del progetto (`format` di Ollama) — il JSON malformato e' strutturalmente impossibile, non serve un retry-loop complesso sul parsing.

**Scope dei dati**: i valori numerici gia' puliti nei CSV grezzi (tasso poverta', conteggio eventi ACLED, flusso migratorio, intensita' conflitto) NON passano da questo notebook — vengono uniti al profilo del nodo direttamente in pandas nel Blocco B. Qui l'LMM produce solo i campi narrativi/qualitativi che richiedono di leggere documenti non strutturati (vedi `config/extraction_schema.json` per il dettaglio).

**Prerequisito**: server Ollama in esecuzione con il modello scaricato:
```
ollama serve &
ollama pull qwen2.5vl:7b
```
Su Leonardo: `module load ollama/<versione>` prima di questi comandi (vedi `scripts_hpc/`).

In [ ]:
import sys
sys.path.append('..')

import json
from jsonschema import Draft202012Validator

from src.extraction import ollama_client, lmm_extractor
from src.extraction.input_assembly import assembla_input
from src.extraction.prompt_builder import costruisci_prompt
from src.utils.config_loader import load_countries, load_extraction_schema

schema = load_extraction_schema()
validator = Draft202012Validator(schema)
paesi = load_countries()
print(f"{len(paesi)} paesi caricati, schema valido: OK")

## Verifica connessione Ollama

Da rifare ogni volta che si cambia macchina (Mac di sviluppo vs nodo GPU su Leonardo): `OLLAMA_HOST` di default e' `http://localhost:11434`, sovrascrivibile con una variabile d'ambiente se il server gira altrove.

In [ ]:
print("Modello atteso:", ollama_client.MODELLO)
print("Host Ollama:", ollama_client.HOST)
try:
    print("Modello disponibile:", ollama_client.modello_disponibile())
except Exception as e:
    print(f"Impossibile connettersi: {e}")
    print("Avvia il server con `ollama serve` (in un altro terminale o job PBS) e riprova.")

## Test su un singolo paese/trimestre

Prima del giro completo (476 combinazioni), verifica su un caso ricco di documenti (Sudan, 2023-Q2 - piena guerra civile, fino a 10 documenti PDF multimodali nel trimestre).

In [ ]:
ISO3_TEST = "SDN"
PERIODO_TEST = "2023-Q2"

input_assemblato = assembla_input(ISO3_TEST, PERIODO_TEST)
prompt, immagini = costruisci_prompt(input_assemblato)
print(f"Prompt: {len(prompt)} caratteri, {len(immagini)} immagini allegate")
print(prompt[:500], "...")

In [ ]:
esito = lmm_extractor.estrai_singolo(ISO3_TEST, PERIODO_TEST, validator, forza=True)
print(esito)

if esito["stato"] == "ok":
    path = lmm_extractor._percorso_output(ISO3_TEST, PERIODO_TEST)
    print(json.dumps(json.load(open(path, encoding="utf-8")), indent=2, ensure_ascii=False))

## Esecuzione completa

Tutti i 17 paesi × 28 trimestri (2018-Q1 — 2024-Q4) = 476 combinazioni. Resumable: se interrotta (walltime PBS scaduto, connessione persa), rilanciare questa stessa cella salta automaticamente le combinazioni gia' estratte e riparte da dove si era fermata.

In [ ]:
riepilogo = lmm_extractor.estrai_tutti()

## Controllo copertura finale

Quanti JSON estratti per paese, su 28 trimestri possibili.

In [ ]:
for p in paesi:
    iso3 = p["iso3"]
    cartella = lmm_extractor.OUTPUT_DIR / iso3
    n = len(list(cartella.glob("*.json"))) if cartella.exists() else 0
    print(f"{iso3}: {n}/28")